Nombre: Daniel Moises Troya Riofrio


Implementar el algoritmo del perceptrón original de Rosenblat para la compuerta AND

In [31]:
import numpy as np

In [32]:
# Función de activación: Escalonada (Heaviside).
# Devuelve 1 si la suma ponderada es positiva o cero, 0 en caso contrario.
# Es ideal para problemas de clasificación lineal binaria.
def escalon(x):
  return 1 if x>=0 else 0

class Perceptron:
    def __init__(self, n_entradas, lr=0.1):
  # Inicialización de pesos: valores pequeños aleatorios para no sesgar el modelo.
  # lr=0.1: Un valor moderado para evitar oscilaciones inestables en el aprendizaje.
        self.w= np.random.randn(n_entradas)*0.5
        self.b= 0.0
        self.lr= lr

    def forward(self,x):
      self.x=x
      self.z= np.dot(self.w,x) + self.b
      self.a = escalon(self.z)
      return self.a

    def backward(self, y_real):
      # Cálculo del error: diferencia entre el valor deseado y la predicción.
      error = y_real - self.a

      # Regla de Rosenblatt: Ajustamos pesos y sesgo sumando el producto del error.
      # Usamos += porque si el error es positivo (deseado 1, obtenido 0),
      # necesitamos incrementar los pesos para que la neurona se active.

      self.w += self.lr * error * self.x
      self.b += self.lr * error

      return error

In [33]:
# Definición de datos de entrada (X) y salida esperada (y) para la compuerta AND
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array( [0,0,0,1])

In [34]:
perceptron=Perceptron(n_entradas=2, lr= 5.0)

In [35]:
# Ciclo de entrenamiento: 100 épocas máximo.
for epoca in range (100):
  error_total = 0
  for i in range(len(X)):
    salida = perceptron.forward(X[i])
    error = perceptron.backward(y[i])
    error_total = error_total + abs(error)

  print("Epoca", epoca)
  print ("Error total", error_total)

  # Criterio de convergencia: Si el error es 0, el perceptrón aprendió perfectamente.
    # El 'break' ahorra recursos de cómputo al detener el ciclo prematuramente.

  if error_total == 0:
    print("Terminado")
    break



Epoca 0
Error total 2
Epoca 1
Error total 3
Epoca 2
Error total 1
Epoca 3
Error total 0
Terminado


In [36]:
# Cálculo matemático para visualizar la línea que separa los datos.
# La ecuación se deriva de: w1*x1 + w2*x2 + b = 0
w1, w2 = perceptron.w
b = perceptron.b
print(f"\nEcuación de la frontera de decisión:")
print(f"x2 = {-w1/w2:.2f} * x1 + {-b/w2:.2f}")


Ecuación de la frontera de decisión:
x2 = -10.32 * x1 + 11.15


Nombre: Daniel Moises Troya Riofrio

Construir una red neuronal multicapa

In [37]:
import numpy as np

In [38]:
from sklearn.datasets import load_iris

In [39]:
# ReLU: Activación estándar para capas ocultas.
# Ayuda a mitigar el desvanecimiento del gradiente.
def relu(x):
  return np.maximum(0, x)

def relu_derivada(x):
  # La derivada de ReLU es 1 si x > 0, 0 en otro caso.
  return (0 > x).astype(float)
# Softmax: Convierte las salidas de la última capa en probabilidades (suman 1)
def softmax(z):
  # Restar el máximo mejora la estabilidad numérica ante números grandes.
  exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

class MLP:
    def __init__(self, capas, lr=0.01):
        self.lr = lr
        self.W= []
        self.b= []

        # Inicialización de pesos usando 'He Initialization' (np.sqrt(2.0/n)),
        # recomendada para redes que usan ReLU para evitar saturación inicial.
        for i in range(len(capas)-1):
          w=np.random.randn(capas[i], capas[i + 1]) * np.sqrt(2.0 / capas[i])
          b= np.zeros((1, capas[i + 1]))
          self.W.append(w)
          self.b.append(b)

    def forward(self,X):
      self.A = [X]
      self.Z = []
      entrada = X

      # Propagación en capas ocultas con ReLU
      for i in range(len(self.W)-1):
        z = np.dot(entrada, self.W[i]) + self.b[i]
        a=relu(z)

        self.Z.append(z)
        self.A.append(a)
        entrada = a
      # Capa de salida con Softmax
      z = np.dot(entrada, self.W[-1]) + self.b[-1]
      a = softmax(z)
      self.Z.append(z)
      self.A.append(a)
      return a

    def backward(self, y_real):
      m=y_real.shape[0]
      # Delta inicial: Error de predicción (Softmax + Cross-Entropy)
      delta = (self.A[-1] - y_real)

      for i in range(len(self.W) - 1, -1, -1):
        dw = np.dot(self.A[i].T, delta) / m
        db = np.sum(delta, axis=0, keepdims=True) / m

        # Propagación del error hacia atrás usando la derivada de ReLU
        if i > 0:
          delta = np.dot(delta, self.W[i].T) * relu_derivada(self.Z[i-1])

        # Actualización de pesos
        self.W[i] -= self.lr * dw
        self.b[i] -= self.lr * db

    def predecir(self, X):
      # Selecciona el índice con la probabilidad más alta
      return np.argmax(self.forward(X), axis=1)

# Carga y Normalización (Z-score)
# Es vital para que las neuronas no se saturen con valores grandes.
iris = load_iris()
X = iris.data
X = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
y = np.eye(3)[iris.target] # One-Hot Encoding para las 3 especies

# Entrenamiento
# Usamos lr=0.1, un compromiso entre velocidad y estabilidad.
red= MLP(capas=[4,8,3], lr= 0.1)
for epocas in range(1000):
    salida = red.forward(X)
    red.backward(y)
    if epocas % 200 == 0:
        # Cross-Entropy Loss
        loss = -np.mean(np.sum(y * np.log(salida + 1e-9), axis=1))
        print("Epoca", epocas)
        print("Loss:", loss)

#Evaluacion
predicciones = red.predecir(X)
accuracy = np.mean(predicciones == iris.target)
print("Accuracy final:", accuracy * 100)

Epoca 0
Loss: 1.4272304578517374
Epoca 200
Loss: 0.28862345814610824
Epoca 400
Loss: 0.2324139345780801
Epoca 600
Loss: 0.21308617380628056
Epoca 800
Loss: 0.2432508035554894
Accuracy final: 90.66666666666666
